# Whitechapel maps: final verified step-by-step run

This notebook runs only the final map dependencies and checks the frozen counts. Coordinates are manual image pixels for the supplied 10,059 × 6,746 Booth scan, not GIS geocodes. Edit coordinate CSVs and rerun; never edit the PNG markers.

In [ ]:
from pathlib import Path
import contextlib, io, os, runpy, sys
import pandas as pd
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / 'run_all.py').exists() and (ROOT.parent / 'run_all.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'run_all.py').exists(), 'Start Jupyter in the reproducibility directory or notebooks/'
os.environ['MPLCONFIGDIR'] = str(ROOT / 'audit' / 'matplotlib')
os.environ['XDG_CACHE_HOME'] = str(ROOT / 'audit' / 'cache')

def run_repository_script(relative_path):
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        result = runpy.run_path(str(ROOT / relative_path), run_name='__main__')
    safe_output = captured.getvalue().replace(str(ROOT), '.')
    if safe_output:
        print(safe_output, end='' if safe_output.endswith('\n') else '\n')
    return result

print('Repository: reproducibility/')
print('Python:', sys.version.split()[0])


In [ ]:
booth = Image.open(ROOT / 'maps' / 'background_map' / 'Booth_Sheet63_Whitechapel_1898_1899.jpg')
assert booth.size == (10059, 6746)
print('Booth scan:', booth.size, '— exact size assertion passed')

## Workplace extraction and Figure 5

In [ ]:
_ = run_repository_script('scripts/19_moh_workplace_address_extraction.py')


In [ ]:
work = pd.read_csv(ROOT / 'outputs' / 'tables' / 'technical' / 'workplace_address_extraction.csv')
address_count = work['address'].nunique()
types = work['workplace_type'].str.lower().replace({'workshops': 'workshop'})
type_counts = types.value_counts().to_dict()
assert (len(work), address_count, type_counts.get('workshop'), type_counts.get('bakehouse')) == (24, 23, 20, 4)
print('24 proceedings')
print('23 unique addresses')
print('20 workshop records')
print('4 bakehouse records')

## Source validation for Figure 6 and place aggregation for Figure 7

In [ ]:
_ = run_repository_script('scripts/16_moh_spatial_disease_validation.py')


In [ ]:
review = pd.read_csv(ROOT / 'analysis' / 'chapter3' / 'spatial_review_log.csv')
retained = review[review['review_status'] != 'exclude_from_substantive_interpretation']
places = sorted({p.strip() for value in retained['matched_places'] for p in str(value).split(';') if p.strip()})
pairs = pd.read_csv(ROOT / 'outputs' / 'tables' / 'technical' / 'spatial_disease_validation.csv')
pair_counts = pairs['validation_status'].value_counts().to_dict()
assert (len(review), len(retained), len(places), len(pairs)) == (38, 32, 16, 12)
assert pair_counts == {'false_structural_adjacency': 9, 'direct_place_institution_disease': 2, 'management_context_only': 1}
print('38 raw spatial records')
print('32 source-valid records')
print('16 named places')
print('12 place–disease pairs: 9 false, 2 direct, 1 management')

## Generate the verified maps from coordinate CSVs

In [ ]:
_ = run_repository_script('scripts/20_whitechapel_maps.py')


In [ ]:
from IPython.display import display
for filename in ('figure_05_workplace_enforcement_map.png', 'figure_07_source_validated_places_map.png'):
    preview = Image.open(ROOT / 'outputs' / 'figures' / filename).copy()
    preview.thumbnail((900, 900), Image.Resampling.LANCZOS)
    print(filename, 'full output retained in outputs/figures; notebook preview:', preview.size)
    display(preview)